# B2 proxy refit (multiclass HGB with class probabilities)

Refits the HGB multiclass model of the 35 benchmark targets on five folds (175 fits) and stores the class
probabilities, so the cost-aware ask rule can be tested with AutoGluon's ordinal decision rule. Every fit is saved to
Drive as soon as it finishes. After a disconnect, run cells 1–3 again and then cell 5: finished fits are skipped.

**scikit-learn 1.9.1 is required.** The frozen fits used it; Colab's default version gives different fits on 21 of the
35 targets. Cell 2 installs it and cell 3 refuses to continue with any other version.

**Runtime:** CPU (the default) is enough. About 1 min for cell 2 and 25–40 min for cell 5.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/task_inference_proxy'   # change if you uploaded elsewhere
RESULTS = BASE + '/results_sklearn191'                 # a new folder: do not reuse the first run's results/
import os
for f in ('config.json', 'targets.csv', 'frozen_multiclass.csv', 'proxy_lib.py', 'data'):
    assert os.path.exists(os.path.join(BASE, f)), f'missing {f}'
print('ok')

## 2. Install scikit-learn 1.9.1 (~1 min)

If Colab asks you to restart the session after this, click **Restart**, then run cell 1 and continue with cell 3.

In [ ]:
%pip install -q "scikit-learn==1.9.1"

## 3. Load the library and check the version

This cell stops with a message if scikit-learn is not 1.9.1. Then use Runtime › Restart session and run cell 1 and
this cell again.

In [ ]:
import sys, importlib
sys.path.insert(0, BASE)
import proxy_lib
importlib.reload(proxy_lib)
cfg, targets = proxy_lib.load_bundle(BASE)
proxy_lib.check_sklearn(cfg)
print(len(targets), 'targets;', cfg)
print(proxy_lib.versions())

## 4. Smoke test (a few seconds, writes to /content)

I010 is one of the targets whose fit depends on the scikit-learn version. The last line must say `1/1 identical`.
If it does not, stop and send the output.

In [ ]:
proxy_lib.run(BASE, '/content/smoke_proxy_191', only={'I010'}, folds=[0])
_ = proxy_lib.summarize('/content/smoke_proxy_191', BASE)

## 5. Full run (~25–40 min, resumable)

In [ ]:
proxy_lib.run(BASE, RESULTS)

## 6. Summary and zip

The fidelity line should say `175/175 identical`. Download `proxy_results_sklearn191.zip` (also saved in the Drive
folder) and put it in the project folder.

In [ ]:
import shutil
_ = proxy_lib.summarize(RESULTS, BASE)
zip_path = shutil.make_archive(BASE + '/proxy_results_sklearn191', 'zip', RESULTS)
print('wrote', zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:  # the zip is also in the Drive folder
    print(f'automatic download failed ({e}); download proxy_results_sklearn191.zip from the Drive folder')

### Optional: retry fits that failed

Only if cell 6 lists errors that look like a disconnect or out-of-memory.

In [ ]:
proxy_lib.run(BASE, RESULTS, retry_errors=True)